# Week 3 Advanced — State Space, Modal Participation, Controllability, Observability

This notebook uses the same flexible drivetrain, but now the point is to connect the matrices to physical modes and sensor/actuator placement.

$$\dot{x}=Ax+Bu, \qquad y=Cx$$

with $x=[\omega_1,\omega_2,\phi]^T$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

J1,J2,k,c=0.05,0.09,10.0,0.08
A=np.array([[-c/J1,c/J1,-k/J1],[c/J2,-c/J2,k/J2],[1,-1,0]])
B=np.array([[1/J1],[0],[0]])
lam,V=np.linalg.eig(A)
print('Eigenvalues:'); print(lam)


## 1. Modal decomposition

For a free response,
$$x(t)=Ve^{\Lambda t}V^{-1}x(0)=\sum_i c_i v_i e^{\lambda_i t}.$$
The eigenvectors tell you the *shape* of each mode. The coefficients $c_i$ tell you how strongly the chosen initial condition excites them.

In [ ]:
x0=np.array([5.0,-3.0,0.02])
coeff=np.linalg.solve(V,x0)
for i,(la,ci) in enumerate(zip(lam,coeff)):
    print(f'Mode {i}: lambda={la:.4f}, coefficient={ci:.4f}')

t=np.linspace(0,2,1000)
X=np.array([expm(A*ti)@x0 for ti in t])
plt.figure(figsize=(9,4)); plt.plot(t,X); plt.legend(['omega1','omega2','phi']); plt.grid(True); plt.show()


## 2. Sensor placement changes observability

Compare three sensors: motor speed only, load speed only, and shaft twist only. Compute the observability matrix rank and condition number. A full rank can still be numerically weak.

In [ ]:
def obsv(A,C): return np.vstack([C,C@A,C@A@A])
for name,C in {
    'motor speed':np.array([[1,0,0.]]),
    'load speed':np.array([[0,1,0.]]),
    'shaft twist':np.array([[0,0,1.]])}.items():
    O=obsv(A,C)
    print(name, 'rank=',np.linalg.matrix_rank(O),'cond=',np.linalg.cond(O))


## 3. Actuator placement changes controllability

Now compare applying torque to inertia 1 versus inertia 2. Then inspect the projections of $B$ into the left-eigenvector basis. That tells you which modes the actuator couples to strongly or weakly.

In [ ]:
def ctrb(A,B): return np.hstack([B,A@B,A@A@B])
for name,Bi in {'motor torque':np.array([[1/J1],[0],[0]]),'load torque':np.array([[0],[1/J2],[0]])}.items():
    Ctr=ctrb(A,Bi)
    print(name,'rank=',np.linalg.matrix_rank(Ctr),'cond=',np.linalg.cond(Ctr))

W=np.linalg.inv(V)
print('Modal input coupling W B:'); print(W@B)


## Engineering tasks

- Find an initial condition that excites the torsional mode strongly but barely excites the rigid-body mode.
- Find one that does the opposite.
- Explain why full rank does **not** mean all states/modes are equally easy to control or estimate.
- Add realistic sensor noise and test whether the most ill-conditioned sensor choice still works well numerically.
- Use the PBH test directly on each eigenvalue.

**Deliverable:** recommend one actuator and one sensor set for this drivetrain and justify the choice using both rank and conditioning/modal coupling.